# 08 - Hyperparameter Tuning with Optuna

### Objective

The objective of this notebook is to optimize the machine learning models using `Optuna`.

Optuna efficiently searches for suitable hyperparameter combinations, while `TimeSeriesSplit` is used during validation to preserve the chronological nature of stock market data.

In [26]:
import json

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [6]:
stocks = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "RELIANCE.NS",
    "TCS.NS",
    "HDFCBANK.NS",
    "INFY.NS"
]

In [7]:
ml_data = {}

for stock in stocks:
    
    df = pd.read_csv(f"../data/ml_ready/{stock}_ml.csv", parse_dates=['Date'])
    
    df = df.sort_values('Date').reset_index(drop=True)
    
    ml_data[stock] = df


In [8]:
# verify
for stock in stocks:
    print(stock, ml_data[stock].shape)

AAPL (2715, 24)
MSFT (2715, 24)
NVDA (2715, 24)
AMZN (2715, 24)
RELIANCE.NS (2664, 24)
TCS.NS (2664, 24)
HDFCBANK.NS (2664, 24)
INFY.NS (2664, 24)


## Feature and Target Selection

Use the same engineered features and target defined during model training.

In [9]:
features = [
    "Daily_Return",
    "Return_Lag_1",
    "Return_Lag_2",
    "Return_Lag_3",
    "SMA_5",
    "SMA_20",
    "SMA_50",
    "EMA_20",
    "Volatility_20",
    "Volume_Change",
    "High_Low_Range",
    "Open_Close_Change",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Hist"
]

target = "Target"

In [10]:
X = {}
y = {}

for stock in stocks:
    
    X[stock] = ml_data[stock][features]
    y[stock] = ml_data[stock][target]

## Chronological Train-Test Split

Use the first **80%** of observations for tuning and reserve the latest **20%** for final evaluation.

In [11]:
X_train = {}
X_test = {}

y_train = {}
y_test = {}

for stock in stocks:
    
    split_idx = int(len(X[stock]) * 0.8)
    
    X_train[stock] = X[stock][:split_idx]
    X_test[stock] = X[stock][split_idx:]
    
    y_train[stock] = y[stock][:split_idx]
    y_test[stock] = y[stock][split_idx:]

## Time-Series Cross-Validation

`TimeSeriesSplit` validates each model on future observations while preserving the chronological order of the data.

In [12]:
tscv = TimeSeriesSplit(n_splits=5)

## Random Forest Optimization

Optuna searches for suitable Random Forest hyperparameters by maximizing the mean cross-validation accuracy.

In [13]:
def objective(trial, stock):
    
    # hyperparameters for random forest
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    
    max_depth = trial.suggest_int('max_depth', 3, 20)
    
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    
    
    model_rf = RandomForestClassifier (
        n_estimators=n_estimators, 
        max_depth= max_depth,
        min_samples_leaf= min_samples_leaf,
        min_samples_split=min_samples_split, 
        random_state=42
    )
    
    score = cross_val_score(model_rf, X_train[stock], y_train[stock], cv= tscv, scoring='accuracy', n_jobs=-1).mean()
    return score

In [14]:
studies = {}

for stock in stocks:
    
    print(f"\nOptimizing {stock}...")

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(
        lambda trial: objective(trial, stock),
        n_trials=50
    )
    
    studies[stock] = study

    


Optimizing AAPL...

Optimizing MSFT...

Optimizing NVDA...

Optimizing AMZN...

Optimizing RELIANCE.NS...

Optimizing TCS.NS...

Optimizing HDFCBANK.NS...

Optimizing INFY.NS...


In [15]:
best_params = {}

for stock in stocks:
    
    best_params[stock] = studies[stock].best_params

## Best Hyperparameters

Display the best Random Forest configuration and cross-validation score identified by Optuna for each stock.

In [16]:
for stock in stocks:

    print(f"\n{stock}")

    print("Best CV Accuracy:", studies[stock].best_value)

    print("Best Parameters:", best_params[stock])


AAPL
Best CV Accuracy: 0.5077348066298343
Best Parameters: {'n_estimators': 485, 'max_depth': 9, 'min_samples_split': 6, 'min_samples_leaf': 8}

MSFT
Best CV Accuracy: 0.5453038674033148
Best Parameters: {'n_estimators': 100, 'max_depth': 13, 'min_samples_split': 8, 'min_samples_leaf': 9}

NVDA
Best CV Accuracy: 0.5171270718232044
Best Parameters: {'n_estimators': 417, 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 10}

AMZN
Best CV Accuracy: 0.5342541436464088
Best Parameters: {'n_estimators': 197, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 9}

RELIANCE.NS
Best CV Accuracy: 0.5149295774647887
Best Parameters: {'n_estimators': 168, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 10}

TCS.NS
Best CV Accuracy: 0.5301408450704226
Best Parameters: {'n_estimators': 144, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 1}

HDFCBANK.NS
Best CV Accuracy: 0.5290140845070422
Best Parameters: {'n_estimators': 201, 'max_depth': 13, 'min_sam

## Train Tuned Random Forest Models

Train a final Random Forest model for each stock using the best hyperparameters identified by Optuna.

In [17]:
tuned_rf_models = {}

for stock in stocks:
    
    model = RandomForestClassifier(
        **best_params[stock],
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train[stock], y_train[stock])
    
    tuned_rf_models[stock] = model
    
    print(f"{stock} tuned model trained.")

AAPL tuned model trained.
MSFT tuned model trained.
NVDA tuned model trained.
AMZN tuned model trained.
RELIANCE.NS tuned model trained.
TCS.NS tuned model trained.
HDFCBANK.NS tuned model trained.
INFY.NS tuned model trained.


## Tuned Model Predictions

Use the tuned Random Forest models to predict stock direction on the unseen test datasets.

In [18]:
tuned_predictions = {}

for stock in stocks:

    tuned_predictions[stock] = tuned_rf_models[stock].predict(X_test[stock])

## Tuned Model Evaluation

Evaluate the tuned models using Accuracy, Precision, Recall, and F1-Score.

In [19]:
tuned_results = []

for stock in stocks:

    y_true = y_test[stock]
    y_pred = tuned_predictions[stock]

    tuned_results.append({
        "Stock": stock,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1_Score": f1_score(
            y_true, y_pred, zero_division=0
        )
    })

tuned_results_df = pd.DataFrame(tuned_results)

tuned_results_df

,Stock,Accuracy,Precision,Recall,F1_Score
0,AAPL,0.462247,0.511628,0.370370,0.429688
1,MSFT,0.464088,0.517588,0.345638,0.414487
2,NVDA,0.502762,0.734375,0.156667,0.258242
3,AMZN,0.484346,0.543478,0.173611,0.263158
4,RELIANCE.NS,0.489681,0.652174,0.053763,0.099338
5,TCS.NS,0.538462,0.506494,0.314516,0.388060
6,HDFCBANK.NS,0.485929,0.529915,0.220641,0.311558
7,INFY.NS,0.487805,0.485185,0.494340,0.489720


## Default vs Tuned Random Forest

Compare the default and Optuna-tuned Random Forest models to determine whether hyperparameter optimization improves performance on unseen test data.

In [20]:
default_results = []

for stock in stocks:
    
    model = RandomForestClassifier(
        n_estimators = 100, 
        random_state=42, 
        n_jobs=-1
    )
    
    model.fit(X_train[stock], y_train[stock])
    
    y_pred = model.predict(X_test[stock])
    
    default_results.append({
        "Stock" : stock,
        "Default Accuracy" : accuracy_score(y_test[stock], y_pred)
    })
    
default_results_df = pd.DataFrame(default_results)
    

In [21]:
comparison_df = default_results_df.merge(
    tuned_results_df[["Stock", "Accuracy"]],
    on="Stock"
)

comparison_df.rename(
    columns={"Accuracy": "Tuned_Accuracy"},
    inplace=True
)

comparison_df["Improvement"] = (comparison_df["Tuned_Accuracy"] - comparison_df["Default Accuracy"]
)

comparison_df

,Stock,Default Accuracy,Tuned_Accuracy,Improvement
0,AAPL,0.484346,0.462247,-0.022099
1,MSFT,0.462247,0.464088,0.001842
2,NVDA,0.510129,0.502762,-0.007366
3,AMZN,0.486188,0.484346,-0.001842
4,RELIANCE.NS,0.521576,0.489681,-0.031895
5,TCS.NS,0.542214,0.538462,-0.003752
6,HDFCBANK.NS,0.469043,0.485929,0.016886
7,INFY.NS,0.495310,0.487805,-0.007505


In [ ]:
with open("../models/best_rf_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

In [27]:
tuned_results_df.to_csv(
    "../results/tuned_rf_results.csv",
    index=False
)

## Conclusion

Optuna with time-series cross-validation was used to optimize the Random Forest models.

Hyperparameter tuning improved test accuracy for some stocks but did not consistently outperform the default Random Forest across all companies. This highlights the difficulty of generalizing short-term stock direction patterns to unseen market periods.

The evaluation results are saved for later comparison and final model selection in StockVision.